# 🚀 MobileNetV3 Transfer Learning — Version Optimisée GPU


In [ ]:
# ── Cellule 1 : Vérification GPU ──────────────────────────────────────────────
import torch
import subprocess

print('=' * 60)
print('🔍  VÉRIFICATION GPU')
print('=' * 60)

cuda_ok = torch.cuda.is_available()
print(f'  CUDA disponible         : {cuda_ok}')

if cuda_ok:
    print(f'  Nom du GPU              : {torch.cuda.get_device_name(0)}')
    print(f'  Mémoire totale          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'  CUDA version            : {torch.version.cuda}')
    print(f'  cuDNN version           : {torch.backends.cudnn.version()}')
    # Active cuDNN benchmark pour choisir l'algo convolution le plus rapide
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    print('  cuDNN benchmark         : ACTIVÉ ✅')
else:
    print('  ⚠️  Aucun GPU détecté — entraînement sur CPU (très lent)')

print('=' * 60)


🔍  VÉRIFICATION GPU
  CUDA disponible         : True
  Nom du GPU              : Tesla T4
  Mémoire totale          : 15.64 GB
  CUDA version            : 12.8
  cuDNN version           : 91002
  cuDNN benchmark         : ACTIVÉ ✅


In [ ]:
# ── Cellule 2 : Montage Google Drive ──────────────────────────────────────────
from google.colab import drive
import time

print('=' * 60)
print('📂  MONTAGE GOOGLE DRIVE')
print('=' * 60)

t0 = time.time()
drive.mount('/content/drive')
print(f'  ✅ Drive monté en {time.time()-t0:.1f}s')
print('=' * 60)


📂  MONTAGE GOOGLE DRIVE
Mounted at /content/drive
  ✅ Drive monté en 29.7s


In [ ]:
# ── Cellule 3 : Copie dataset en local (I/O rapide) ───────────────────────────
import shutil, os, time

print('=' * 60)
print('📁  COPIE DU DATASET EN LOCAL')
print('=' * 60)

src  = '/content/drive/MyDrive/splitdataset'
dst  = '/content/dataset'

if os.path.exists(dst):
    print(f'  Dossier {dst} déjà présent — copie ignorée.')
else:
    print(f'  Source  : {src}')
    print(f'  Dest    : {dst}')
    print('  Copie en cours...')
    t0 = time.time()
    shutil.copytree(src, dst)
    print(f'  ✅ Copie terminée en {time.time()-t0:.1f}s')

# Compter les images par split
data_dir = dst
for split in ['train', 'val']:
    split_path = os.path.join(data_dir, split)
    if os.path.exists(split_path):
        classes = os.listdir(split_path)
        n_imgs  = sum(len(os.listdir(os.path.join(split_path, c))) for c in classes)
        print(f'  {split:5s} → {len(classes)} classes | {n_imgs} images')

print('=' * 60)


📁  COPIE DU DATASET EN LOCAL
  Source  : /content/drive/MyDrive/splitdataset
  Dest    : /content/dataset
  Copie en cours...
  ✅ Copie terminée en 1460.1s
  train → 6 classes | 42001 images
  val   → 6 classes | 9016 images


In [ ]:
# ── Cellule 4 : Data pipeline optimisé GPU ────────────────────────────────────
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import multiprocessing

print('=' * 60)
print('⚙️   CONFIGURATION DATA PIPELINE')
print('=' * 60)

# Nombre de workers = nombre de CPU dispo (max 8)
NUM_WORKERS = min(multiprocessing.cpu_count(), 8)
BATCH_SIZE  = 64          # Augmenté : remplit mieux la VRAM
PIN_MEMORY  = True        # Transfert CPU→GPU en mémoire épinglée (plus rapide)
PREFETCH    = 4           # Prefetch factor pour alimenter le GPU sans attente

print(f'  Batch size              : {BATCH_SIZE}')
print(f'  Num workers             : {NUM_WORKERS}')
print(f'  Pin memory              : {PIN_MEMORY}')
print(f'  Prefetch factor         : {PREFETCH}')

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(f'{data_dir}/train', transform=train_transforms)
val_dataset   = datasets.ImageFolder(f'{data_dir}/val',   transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    prefetch_factor=PREFETCH, persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    prefetch_factor=PREFETCH, persistent_workers=True
)

print(f'  Classes ({len(train_dataset.classes)})            : {train_dataset.classes}')
print(f'  Batches train           : {len(train_loader)}')
print(f'  Batches val             : {len(val_loader)}')
print('  ✅ DataLoaders prêts')
print('=' * 60)


⚙️   CONFIGURATION DATA PIPELINE
  Batch size              : 64
  Num workers             : 2
  Pin memory              : True
  Prefetch factor         : 4
  Classes (6)            : ['Anthracnose', 'Ash weevil', 'Canker', 'Greening', 'Healthy', 'Leaf miner']
  Batches train           : 657
  Batches val             : 141
  ✅ DataLoaders prêts


In [ ]:
# ── Cellule 5 : Modèle MobileNetV3 ────────────────────────────────────────────
import torchvision.models as models
import torch.nn as nn

print('=' * 60)
print('🧠  CHARGEMENT MODÈLE MOBILENETV3-LARGE')
print('=' * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'  Device                  : {device}')

model = models.mobilenet_v3_large(weights='IMAGENET1K_V1')

# Gel de toutes les couches sauf classifier
for param in model.parameters():
    param.requires_grad = False

num_classes = len(train_dataset.classes)
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)

# Compte les paramètres
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

model = model.to(device)

print(f'  Paramètres totaux       : {total_params:,}')
print(f'  Paramètres entraînables : {trainable_params:,}')
print(f'  Nouvelles sorties       : {num_classes} classes')

if torch.cuda.is_available():
    mem_alloc = torch.cuda.memory_allocated(0) / 1e6
    print(f'  VRAM après chargement   : {mem_alloc:.1f} MB')

print('  ✅ Modèle chargé sur', device)
print('=' * 60)


🧠  CHARGEMENT MODÈLE MOBILENETV3-LARGE
  Device                  : cuda
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 135MB/s]


  Paramètres totaux       : 4,209,718
  Paramètres entraînables : 7,686
  Nouvelles sorties       : 6 classes
  VRAM après chargement   : 17.0 MB
  ✅ Modèle chargé sur cuda


In [ ]:
# ── Cellule 6 : Fonctions train / eval avec AMP (mixed precision) ─────────────
from torch.cuda.amp import autocast, GradScaler
import time

scaler = GradScaler()   # Gère les gradients en float16 pour économiser la VRAM

def gpu_stats():
    """Retourne une chaîne avec l'utilisation GPU actuelle."""
    if not torch.cuda.is_available():
        return ''
    alloc    = torch.cuda.memory_allocated(0)  / 1e6
    reserved = torch.cuda.memory_reserved(0)   / 1e6
    return f'VRAM {alloc:.0f}/{reserved:.0f} MB'


def train_one_epoch(model, loader, optimizer, criterion, epoch, total_epochs):
    model.train()
    total_loss  = 0.0
    correct     = 0
    total       = 0
    t_start     = time.time()

    for batch_idx, (images, labels) in enumerate(loader):
        # Transfert non-bloquant CPU→GPU
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)   # Plus rapide que zero_grad()

        # ── Mixed Precision (float16) ──
        with autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds       = outputs.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        # Print progression toutes les 10 batches
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(loader):
            elapsed    = time.time() - t_start
            imgs_per_s = (batch_idx + 1) * images.size(0) / elapsed
            print(
                f'    Epoch {epoch}/{total_epochs}'
                f' | Batch {batch_idx+1:3d}/{len(loader)}'
                f' | Loss {total_loss/(batch_idx+1):.4f}'
                f' | Train Acc {correct/total:.3f}'
                f' | {imgs_per_s:.0f} img/s'
                f' | {gpu_stats()}'
            )

    epoch_time = time.time() - t_start
    return total_loss / len(loader), correct / total, epoch_time


def evaluate(model, loader):
    model.eval()
    correct = 0
    total   = 0
    t_start = time.time()

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast():
                outputs = model(images)
            preds    = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_time = time.time() - t_start
    return correct / total, val_time

print('  ✅ Fonctions train/eval définies avec AMP (float16)')


  ✅ Fonctions train/eval définies avec AMP (float16)


/tmp/ipykernel_3035/2171682761.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()   # Gère les gradients en float16 pour économiser la VRAM


In [ ]:
# ── Cellule 7 : Entraînement complet ──────────────────────────────────────────
import torch.optim as optim

EPOCHS = 5
LR     = 1e-3

criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer  = optim.Adam(model.classifier.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history    = []
best_acc   = 0.0
total_t    = time.time()

print('=' * 60)
print('🏋️   DÉMARRAGE ENTRAÎNEMENT')
print('=' * 60)
print(f'  Epochs                  : {EPOCHS}')
print(f'  Learning rate initiale  : {LR}')
print(f'  Scheduler               : CosineAnnealingLR')
print(f'  Loss                    : CrossEntropy + label smoothing 0.1')
print(f'  Mixed Precision (AMP)   : ✅ activé')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]['lr']
    print(f'\n📌 Epoch {epoch}/{EPOCHS}  —  LR = {current_lr:.6f}')
    print(f'  ▶ Phase TRAIN ...')

    train_loss, train_acc, t_train = train_one_epoch(
        model, train_loader, optimizer, criterion, epoch, EPOCHS
    )

    print(f'  ▶ Phase VAL ...')
    val_acc, t_val = evaluate(model, val_loader)

    scheduler.step()

    # Sauvegarde du meilleur modèle
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pth')
        best_tag = '  ⭐ NOUVEAU MEILLEUR MODÈLE SAUVEGARDÉ'
    else:
        best_tag = ''

    history.append({'epoch': epoch, 'loss': train_loss,
                    'train_acc': train_acc, 'val_acc': val_acc})

    if torch.cuda.is_available():
        vram_peak = torch.cuda.max_memory_allocated(0) / 1e6
        torch.cuda.reset_peak_memory_stats(0)
        vram_str  = f'  | VRAM peak {vram_peak:.0f} MB'
    else:
        vram_str = ''

    print(
        f'\n  ✅ Epoch {epoch:2d} résumé'
        f' | Loss {train_loss:.4f}'
        f' | Train Acc {train_acc:.4f}'
        f' | Val Acc {val_acc:.4f}'
        f' | Train {t_train:.1f}s'
        f' | Val {t_val:.1f}s'
        f'{vram_str}'
        f'{best_tag}'
    )

print('\n' + '=' * 60)
print(f'  🏁 Entraînement terminé en {time.time()-total_t:.1f}s')
print(f'  🏆 Meilleure Val Acc : {best_acc:.4f}')
print(f'  💾 Meilleur modèle  : /content/best_model.pth')
print('=' * 60)


🏋️   DÉMARRAGE ENTRAÎNEMENT
  Epochs                  : 5
  Learning rate initiale  : 0.001
  Scheduler               : CosineAnnealingLR
  Loss                    : CrossEntropy + label smoothing 0.1
  Mixed Precision (AMP)   : ✅ activé

📌 Epoch 1/5  —  LR = 0.001000
  ▶ Phase TRAIN ...


/tmp/ipykernel_3035/2171682761.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


    Epoch 1/5 | Batch  10/657 | Loss 1.5876 | Train Acc 0.394 | 56 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch  20/657 | Loss 1.4501 | Train Acc 0.489 | 92 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch  30/657 | Loss 1.3623 | Train Acc 0.540 | 109 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch  40/657 | Loss 1.2939 | Train Acc 0.580 | 124 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch  50/657 | Loss 1.2473 | Train Acc 0.602 | 139 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch  60/657 | Loss 1.2080 | Train Acc 0.622 | 150 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch  70/657 | Loss 1.1737 | Train Acc 0.639 | 158 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch  80/657 | Loss 1.1515 | Train Acc 0.649 | 161 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch  90/657 | Loss 1.1332 | Train Acc 0.655 | 165 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch 100/657 | Loss 1.1133 | Train Acc 0.665 | 172 img/s | VRAM 74/436 MB
    Epoch 1/5 | Batch 110/657 | Loss 1.0994 | Train Acc 0.671 | 178 img/s | VRAM 74/436 MB
 

/tmp/ipykernel_3035/2171682761.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



  ✅ Epoch  1 résumé | Loss 0.9136 | Train Acc 0.7705 | Val Acc 0.8271 | Train 191.7s | Val 22.7s  | VRAM peak 589 MB  ⭐ NOUVEAU MEILLEUR MODÈLE SAUVEGARDÉ

📌 Epoch 2/5  —  LR = 0.000905
  ▶ Phase TRAIN ...
    Epoch 2/5 | Batch  10/657 | Loss 0.8204 | Train Acc 0.839 | 241 img/s | VRAM 74/449 MB
    Epoch 2/5 | Batch  20/657 | Loss 0.8413 | Train Acc 0.820 | 199 img/s | VRAM 74/449 MB
    Epoch 2/5 | Batch  30/657 | Loss 0.8321 | Train Acc 0.820 | 218 img/s | VRAM 74/449 MB
    Epoch 2/5 | Batch  40/657 | Loss 0.8379 | Train Acc 0.816 | 230 img/s | VRAM 74/449 MB
    Epoch 2/5 | Batch  50/657 | Loss 0.8367 | Train Acc 0.816 | 241 img/s | VRAM 74/449 MB
    Epoch 2/5 | Batch  60/657 | Loss 0.8398 | Train Acc 0.812 | 246 img/s | VRAM 74/449 MB
    Epoch 2/5 | Batch  70/657 | Loss 0.8397 | Train Acc 0.812 | 242 img/s | VRAM 74/449 MB
    Epoch 2/5 | Batch  80/657 | Loss 0.8388 | Train Acc 0.812 | 233 img/s | VRAM 74/449 MB
    Epoch 2/5 | Batch  90/657 | Loss 0.8353 | Train Acc 0.814 | 2

In [ ]:
# ── Cellule 8 : Résumé final + copie vers Drive ───────────────────────────────
import shutil

print('=' * 60)
print('📊  RÉSUMÉ FINAL')
print('=' * 60)
print(f'  {"Epoch":>5}  {"Loss":>8}  {"Train Acc":>10}  {"Val Acc":>10}')
print('  ' + '-' * 40)
for h in history:
    best_mark = ' ⭐' if h['val_acc'] == best_acc else ''
    print(f'  {h["epoch"]:>5}  {h["loss"]:>8.4f}  {h["train_acc"]:>10.4f}  {h["val_acc"]:>10.4f}{best_mark}')

# Sauvegarde aussi vers Drive
drive_save = '/content/drive/MyDrive/best_model.pth'
shutil.copy('/content/best_model.pth', drive_save)
print(f'\n  ✅ Modèle copié vers Google Drive : {drive_save}')
print('=' * 60)


📊  RÉSUMÉ FINAL
  Epoch      Loss   Train Acc     Val Acc
  ----------------------------------------
      1    0.9136      0.7705      0.8271
      2    0.8343      0.8158      0.8487
      3    0.8298      0.8164      0.8537
      4    0.8140      0.8261      0.8588
      5    0.8088      0.8318      0.8611 ⭐

  ✅ Modèle copié vers Google Drive : /content/drive/MyDrive/best_model.pth


In [ ]:
# ── TEST : Évaluation complète — images depuis Google Drive ───────────────────
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
import numpy as np
import time

# ── 1. MONTAGE DRIVE ───────────────────────────────────────────────────────────
from google.colab import drive

print('=' * 62)
print('📂  MONTAGE GOOGLE DRIVE')
print('=' * 62)

drive.mount('/content/drive')
print('  ✅ Drive monté')

# ── 2. CONFIG ──────────────────────────────────────────────────────────────────
# 👇 Modifier ce chemin selon ton organisation Drive
TEST_DIR   = '/content/drive/MyDrive/splitdataset/test'
MODEL_PATH = '/content/drive/MyDrive/best_model.pth'
BATCH_SIZE = 32       # Réduit car les I/O Drive sont plus lentes
NUM_WORKERS = 2       # Réduit pour éviter les timeouts Drive

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('\n' + '=' * 62)
print('🧪  ÉVALUATION SUR LE JEU DE TEST')
print('=' * 62)
print(f'  Device     : {device}')
print(f'  Test dir   : {TEST_DIR}')
print(f'  Modèle     : {MODEL_PATH}')
print(f'  Batch size : {BATCH_SIZE}')

# ── 3. DATASET TEST DEPUIS DRIVE ───────────────────────────────────────────────
print('\n📂  Chargement du jeu de test depuis Drive...')

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transforms)
test_loader  = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

class_names = test_dataset.classes
num_classes = len(class_names)
print(f'  Classes ({num_classes})  : {class_names}')
print(f'  Images      : {len(test_dataset)}')
print(f'  Batches     : {len(test_loader)}')

# ── 4. CHARGEMENT DU MODÈLE DEPUIS DRIVE ──────────────────────────────────────
print('\n🧠  Chargement du modèle depuis Drive...')

model = models.mobilenet_v3_large(weights=None)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

print(f'  ✅ Poids chargés depuis {MODEL_PATH}')

# ── 5. INFÉRENCE ───────────────────────────────────────────────────────────────
print('\n⚙️   Inférence en cours...')

all_preds  = []
all_labels = []
all_probs  = []
correct    = 0
total      = 0
t_start    = time.time()

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast():
            outputs = model(images)

        probs  = torch.softmax(outputs, dim=1)
        preds  = outputs.argmax(1)

        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        all_preds  .extend(preds.cpu().numpy())
        all_labels .extend(labels.cpu().numpy())
        all_probs  .extend(probs.cpu().numpy())

        if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == len(test_loader):
            acc     = correct / total
            elapsed = time.time() - t_start
            print(f'    Batch {batch_idx+1:3d}/{len(test_loader)} '
                  f'| Acc : {acc:.4f} '
                  f'| Temps : {elapsed:.1f}s')

elapsed  = time.time() - t_start
test_acc = correct / total

# ── 6. RÉSULTATS GLOBAUX ───────────────────────────────────────────────────────
print('\n' + '=' * 62)
print('📊  RÉSULTATS GLOBAUX')
print('=' * 62)
print(f'  Accuracy globale : {test_acc:.4f}  ({correct}/{total})')
print(f'  Temps total      : {elapsed:.1f}s  ({total/elapsed:.0f} img/s)')

# ── 7. ACCURACY PAR CLASSE ────────────────────────────────────────────────────
print('\n📋  ACCURACY PAR CLASSE')
print(f'  {"Classe":<20} {"Correct":>8} {"Total":>8} {"Acc":>8}')
print('  ' + '-' * 48)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

for i, cls in enumerate(class_names):
    mask    = all_labels == i
    n_total = mask.sum()
    n_ok    = (all_preds[mask] == i).sum()
    acc_cls = n_ok / n_total if n_total > 0 else 0
    bar     = '█' * int(acc_cls * 20) + '░' * (20 - int(acc_cls * 20))
    print(f'  {cls:<20} {n_ok:>8} {n_total:>8} {acc_cls:>7.4f}  {bar}')

# ── 8. MATRICE DE CONFUSION ───────────────────────────────────────────────────
print('\n🔢  MATRICE DE CONFUSION')
print(f'  {"":>16}', end='')
for cls in class_names:
    print(f'  {cls[:8]:>8}', end='')
print()
print('  ' + '-' * (18 + 10 * num_classes))

for i, cls_true in enumerate(class_names):
    print(f'  {cls_true[:16]:<16}', end='')
    for j in range(num_classes):
        mask  = all_labels == i
        count = (all_preds[mask] == j).sum()
        marker = f'[{count:>5}]' if i == j else f' {count:>6} '
        print(f'  {marker}', end='')
    print()

# ── 9. TOP ERREURS ─────────────────────────────────────────────────────────────
print('\n❌  CONFUSIONS LES PLUS FRÉQUENTES (top 5)')
print(f'  {"Vrai":>20}  →  {"Prédit":<20}  {"Nb":>6}')
print('  ' + '-' * 54)

confusion = {}
for true, pred in zip(all_labels, all_preds):
    if true != pred:
        key = (class_names[true], class_names[pred])
        confusion[key] = confusion.get(key, 0) + 1

for (true_cls, pred_cls), count in sorted(confusion.items(), key=lambda x: -x[1])[:5]:
    print(f'  {true_cls:>20}  →  {pred_cls:<20}  {count:>6}')

print('\n' + '=' * 62)
print(f'  ✅ Test terminé — Accuracy finale : {test_acc:.4f}')
print('=' * 62)

📂  MONTAGE GOOGLE DRIVE
Mounted at /content/drive
  ✅ Drive monté

🧪  ÉVALUATION SUR LE JEU DE TEST
  Device     : cuda
  Test dir   : /content/drive/MyDrive/splitdataset/test
  Modèle     : /content/drive/MyDrive/best_model.pth
  Batch size : 32

📂  Chargement du jeu de test depuis Drive...
  Classes (6)  : ['Anthracnose', 'Ash weevil', 'Canker', 'Greening', 'Healthy', 'Leaf miner']
  Images      : 9000
  Batches     : 282

🧠  Chargement du modèle depuis Drive...
  ✅ Poids chargés depuis /content/drive/MyDrive/best_model.pth

⚙️   Inférence en cours...


/tmp/ipykernel_8952/3133164800.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


    Batch   5/282 | Acc : 0.8500 | Temps : 54.1s
    Batch  10/282 | Acc : 0.8594 | Temps : 54.9s
    Batch  15/282 | Acc : 0.8500 | Temps : 55.9s
    Batch  20/282 | Acc : 0.8516 | Temps : 56.6s
    Batch  25/282 | Acc : 0.8237 | Temps : 57.6s
    Batch  30/282 | Acc : 0.8000 | Temps : 58.5s
    Batch  35/282 | Acc : 0.8089 | Temps : 59.6s
    Batch  40/282 | Acc : 0.8086 | Temps : 60.4s
    Batch  45/282 | Acc : 0.8028 | Temps : 61.6s
    Batch  50/282 | Acc : 0.8056 | Temps : 115.8s
    Batch  55/282 | Acc : 0.8227 | Temps : 174.0s
    Batch  60/282 | Acc : 0.8344 | Temps : 252.9s
    Batch  65/282 | Acc : 0.8462 | Temps : 311.4s
    Batch  70/282 | Acc : 0.8549 | Temps : 390.9s
    Batch  75/282 | Acc : 0.8642 | Temps : 447.3s
    Batch  80/282 | Acc : 0.8695 | Temps : 528.8s
    Batch  85/282 | Acc : 0.8761 | Temps : 587.1s
    Batch  90/282 | Acc : 0.8802 | Temps : 665.3s
    Batch  95/282 | Acc : 0.8852 | Temps : 725.0s
    Batch 100/282 | Acc : 0.8834 | Temps : 802.4s
    Batch